In [5]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/pragmatics_project/results/merged_clean.csv")

# restrict to core contrast
sub = df[
    (df["step"] == "inference_derivation")
    & (df["condition"].isin(["critical", "baseline_ordinary"]))
].copy()

# stage as ordered categorical, pretrained as reference level
sub["stage"] = pd.Categorical(sub["stage"], categories=["pretrained", "sft", "dpo"], ordered=False)
sub["condition"] = pd.Categorical(sub["condition"], categories=["baseline_ordinary", "critical"], ordered=False)

print(f"N observations in model: {len(sub)} (expect 24 items x 2 conditions x 3 checkpoints = 144)")

# fixed effects formula with interaction; random intercept per item
model = BinomialBayesMixedGLM.from_formula(
    "outcome_likely ~ stage * condition",
    vc_formulas={'item_id': '1'},
    data=sub,
)
result = model.fit_vb()  # variational Bayes fit
print(result.summary())

N observations in model: 144 (expect 24 items x 2 conditions x 3 checkpoints = 144)
                           Binomial Mixed GLM Results
                                   Type Post. Mean Post. SD   SD  SD (LB) SD (UB)
---------------------------------------------------------------------------------
Intercept                             M    -0.4401   0.2015                      
stage[T.sft]                          M    -1.1321   0.3563                      
stage[T.dpo]                          M    -1.8036   0.4263                      
condition[T.critical]                 M     0.2764   0.2743                      
stage[T.sft]:condition[T.critical]    M     0.0856   0.4761                      
stage[T.dpo]:condition[T.critical]    M     0.1590   0.5500                      
item_id                               V    -0.6620   0.7724 0.516   0.110   2.418
Parameter types are mean structure (M) and variance structure (V)
Variance parameters are modeled as log standard deviations